In [ ]:
!pip install yfinance altair polars statsforecast neuralforecast "vegafusion[embed]>=1.5.0" "vl-convert-python>=1.6.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 30.0 MB/s eta 0:00:00


# Long-Term Stock Forecasting
Following the CRISP-DM methodology, this notebook explores 5-year historical stock data to produce a 1-month (30-day) forecast. We compare traditional statistical baselines (ARIMA/SARIMA) and deep learning baselines (LSTM) against advanced models from the Nixtla ecosystem.

## *1. Business & Data Understanding*
The goal is to provide a 1-month forecast of daily stock prices to inform long-term trading strategies. We evaluate various modeling techniques to determine their predictive capability over a one-month horizon.

In [ ]:
import yfinance as yf
import altair as alt
import polars as pl
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SeasonalNaive
from neuralforecast.models import LSTM, NHITS
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MAE
import os

os.environ['NIXTLA_ID_AS_COL'] = '1'

## *2. Configurations*
Define the stock tickers, historical data period, and the forecast horizon.

In [ ]:
STOCKS = [
    "BBCA.JK",
    "BREN.JK",
    "TPIA.JK",
    "DCII.JK",
    "ASII.JK",
    "SRAJ.JK",
    "HMSP.JK",
]
PERIOD = "5y"
HORIZON = 30  # 1 month forecast

## *3. Data Preparation*
We download the data via `yfinance`, flatten the multi-index, and convert the dataset into the format required by Nixtla (`unique_id`, `ds`, `y`). We then split the data into training and test sets.

In [ ]:
df_pd = yf.download(STOCKS, period=PERIOD)

# Flatten multi-index
df_flat = df_pd.stack(level=1, future_stack=True).reset_index()

# Rename columns to match Nixtla requirements (ds, unique_id, y)
nixtla_df = df_flat[['Date', 'Ticker', 'Close']].rename(
    columns={'Date': 'ds', 'Ticker': 'unique_id', 'Close': 'y'}
)

# Handle missing values and convert to datetime
nixtla_df = nixtla_df.dropna(subset=['y']).copy()
nixtla_df['ds'] = pd.to_datetime(nixtla_df['ds']).dt.tz_localize(None)

# Train/Test Split (Leave last HORIZON days for test)
train_df = (
    nixtla_df.groupby('unique_id', group_keys=False)
    .apply(lambda x: x.iloc[:-HORIZON])
    .reset_index(drop=True)
)
test_df = (
    nixtla_df.groupby('unique_id', group_keys=False)
    .apply(lambda x: x.iloc[-HORIZON:])
    .reset_index(drop=True)
)

# For display
nixtla_df.head()

/tmp/ipykernel_10683/1884807241.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_pd = yf.download(STOCKS, period=PERIOD)
[*********************100%***********************]  8 of 8 completed
/tmp/ipykernel_10683/1884807241.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[:-HORIZON])
/tmp/ipykernel_10683/1884807241.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby t

Price,ds,unique_id,y
0,2021-05-21,ASII.JK,3422.584473
1,2021-05-21,BBCA.JK,5449.503906
3,2021-05-21,BYAN.JK,1110.564209
4,2021-05-21,DCII.JK,11500.000000
5,2021-05-21,HMSP.JK,881.473633


## *4. Exploratory Analysis*
Visualizing the historical closing prices over the 5-year period using Altair.

In [ ]:
alt.data_transformers.enable("vegafusion")

chart = (
    alt.Chart(nixtla_df)
    .mark_line()
    .encode(
        x=alt.X('ds:T', title='Date'),
        y=alt.Y('y:Q', title='Closing Price'),
        color=alt.Color('unique_id:N', title='Ticker'),
        tooltip=['ds:T', 'unique_id:N', 'y:Q'],
    )
    .properties(width='container', height=400, title="Historical Closing Prices")
    .interactive()
)

chart

alt.Chart(...)

## *5. Modeling*
We utilize the `StatsForecast` library for traditional models (AutoARIMA and SeasonalNaive) and `NeuralForecast` for deep learning models (LSTM as a baseline and NHITS as an advanced comparable).

### StatsForecast Modeling

In [ ]:
sf = StatsForecast(
    models=[
        AutoARIMA(season_length=5),  # Assuming 5 trading days in a week
        SeasonalNaive(season_length=5),
    ],
    freq='B',  # Business days
    n_jobs=-1,
)
sf.fit(train_df)
sf_forecast = sf.predict(h=HORIZON).reset_index()
sf_forecast.head()

,index,unique_id,ds,AutoARIMA,SeasonalNaive
0,0,ASII.JK,2026-04-09,5853.879504,5949.588379
1,1,ASII.JK,2026-04-10,5821.319274,5783.000000
2,2,ASII.JK,2026-04-13,5828.907312,5806.798340
3,3,ASII.JK,2026-04-14,5847.601424,5616.411621
4,4,ASII.JK,2026-04-15,5858.016415,5901.991699


### NeuralForecast Modeling

In [ ]:
nf_models = [
    LSTM(
        h=HORIZON,
        max_steps=100,
        scaler_type='standard',
        encoder_hidden_size=64,
        decoder_hidden_size=64,
        loss=MAE(),
    ),
    NHITS(h=HORIZON, input_size=3 * HORIZON, max_steps=100, scaler_type='standard', loss=MAE()),
]

nf = NeuralForecast(models=nf_models, freq='B')
nf.fit(df=train_df)
nf_forecast = nf.predict().reset_index()
nf_forecast.head()

/usr/local/lib/python3.12/dist-packages/neuralforecast/common/_base_model.py:152: UserWarning: Input size too small. Automatically setting input size to 3 * horizon = 90
  warnings.warn(
INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | hist_encoder | LSTM          | 50.4 K | train
4 | mlp_decoder  | MLP           | 4.2 K  | train
-------------------------------------------------------
54.7 K    Trainable params
0         Non-trainable params
54.7 K    Total params
0.219    

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Predicting: |          | 0/? [00:00<?, ?it/s]

,index,unique_id,ds,LSTM,NHITS
0,0,ASII.JK,2026-04-09,6372.213379,5911.375977
1,1,ASII.JK,2026-04-10,6287.276367,5914.154297
2,2,ASII.JK,2026-04-13,6198.472168,5913.666992
3,3,ASII.JK,2026-04-14,6173.748535,5928.998535
4,4,ASII.JK,2026-04-15,6148.166992,5949.665039


## *6. Evaluation*
We merge the forecasts from all models with our test set and evaluate their performance based on the Mean Absolute Percentage Error (MAPE).

In [ ]:
# Merge forecasts
forecasts = test_df.merge(sf_forecast, on=['unique_id', 'ds'], how='left')
forecasts = forecasts.merge(nf_forecast, on=['unique_id', 'ds'], how='left')

# Fill NaN values with ffill in case of mismatched dates
forecasts = forecasts.ffill().bfill()

# Calculate Evaluation Metrics
def calculate_mape(y_true, y_pred):
    return (abs((y_true - y_pred) / y_true)).mean() * 100


def calculate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE (%)': calculate_mape(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
    }

metrics = []
models_to_eval = ['AutoARIMA', 'SeasonalNaive', 'LSTM', 'NHITS']
for model_name in models_to_eval:
    model_metrics = calculate_metrics(forecasts['y'], forecasts[model_name])
    model_metrics['Model'] = model_name
    metrics.append(model_metrics)

metrics_df = pd.DataFrame(metrics).sort_values('RMSE')
metrics_df

,Model,MAPE (%)
0,AutoARIMA,9.728982
1,SeasonalNaive,10.213545
3,NHITS,13.333174
2,LSTM,14.808479


In [ ]:
# Visualization of Forecasts vs Actuals
plot_df = forecasts.melt(
    id_vars=['unique_id', 'ds'],
    value_vars=['y', 'AutoARIMA', 'LSTM', 'NHITS'],
    var_name='Model',
    value_name='Price',
)

eval_charts = []
for uid in plot_df['unique_id'].unique():
    uid_df = plot_df[plot_df['unique_id'] == uid]
    c = (
        alt.Chart(uid_df)
        .mark_line()
        .encode(
            x=alt.X('ds:T', title='Date'),
            y=alt.Y('Price:Q', title='Price'),
            color='Model:N',
            tooltip=['ds:T', 'Model:N', 'Price:Q'],
        )
        .properties(title=f"Forecast Evaluation for {uid}", width=800, height=300)
        .interactive()
    )
    eval_charts.append(c)

alt.vconcat(*eval_charts)

alt.VConcatChart(...)